## Text Normalization

In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

text = "Experienced Python Developer!!! Worked 5+ years at Google."

text = text.lower()
text = re.sub(r'[^a-z\s]', '', text)

tokens = text.split()
tokens = [word for word in tokens if word not in stopwords.words('english')]

lemmatizer = WordNetLemmatizer()
tokens = [lemmatizer.lemmatize(word) for word in tokens]

normalized_text = " ".join(tokens)
print(normalized_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


experienced python developer worked year google


## NER (Named Entity Recognition)

In [1]:
import spacy

In [2]:
nlp = spacy.load("en_core_web_sm")

text = "Dev worked at Google and studied at IIT Roorkee in 2023."

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, ent.label_)

Dev PERSON
Google ORG
IIT Roorkee ORG
2023 DATE


## ATS scoring testing

In [1]:
import re
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [2]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove numbers & punctuation
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)

In [3]:
def exact_keyword_score(job_text, resume_text):
    job_tokens = set(job_text.split())
    resume_tokens = set(resume_text.split())

    if len(job_tokens) == 0:
        return 0

    match_count = len(job_tokens.intersection(resume_tokens))
    return match_count / len(job_tokens)

In [4]:
def job_title_score(job_title, resume_title):
    job_title = normalize_text(job_title)
    resume_title = normalize_text(resume_title)

    return 1.0 if job_title in resume_title else 0.0

In [5]:
def cosine_score(text1, text2):
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([text1, text2])
    similarity = cosine_similarity(vectors[0:1], vectors[1:2])
    return similarity[0][0]

In [7]:
WEIGHTS = {
    "exact_keyword": 0.35,
    "job_title": 0.30,
    "semantic_similarity": 0.20,
    "summary_similarity": 0.15
}

def compute_ats_score(job_json, resume_json):

    # Normalize texts
    job_desc = normalize_text(job_json["description"])
    resume_exp = normalize_text(resume_json["experience"])
    resume_summary = normalize_text(resume_json["summary"])

    # 1. Exact Keyword Match
    exact_score = exact_keyword_score(job_desc, resume_exp)

    # 2. Job Title Match
    title_score = job_title_score(job_json["title"], resume_json["current_title"])

    # 3. Semantic Similarity (Full Description vs Experience)
    semantic_sim = cosine_score(job_desc, resume_exp)

    # 4. Professional Summary Fit
    summary_sim = cosine_score(job_desc, resume_summary)

    # Final Weighted Score
    final_score = (
        WEIGHTS["exact_keyword"] * exact_score +
        WEIGHTS["job_title"] * title_score +
        WEIGHTS["semantic_similarity"] * semantic_sim +
        WEIGHTS["summary_similarity"] * summary_sim
    )

    return {
        "exact_keyword_score": round(exact_score, 4),
        "job_title_score": round(title_score, 4),
        "semantic_similarity": round(semantic_sim, 4),
        "summary_similarity": round(summary_sim, 4),
        "final_ats_score": round(final_score, 4)
    }

In [8]:
if __name__ == "__main__":

    job_json = {
        "title": "Python Backend Developer",
        "description": "Looking for python developer with experience in flask, api development, and database systems."
    }

    resume_json = {
        "current_title": "Backend Developer",
        "summary": "Experienced python backend engineer with strong api development knowledge.",
        "experience": "Worked on python flask applications, built rest api systems and managed database integration."
    }

    result = compute_ats_score(job_json, resume_json)

    print(json.dumps(result, indent=4))

{
    "exact_keyword_score": 0.5556,
    "job_title_score": 0.0,
    "semantic_similarity": 0.3391,
    "summary_similarity": 0.2169,
    "final_ats_score": 0.2948
}
